# 2025 Race Telemetry Bad-Lap EDA

This notebook audits 2025 race-session telemetry quality in the local
Race Telemetry Workbench import and rewrites the analysis as a
narrative walkthrough instead of a table dump.

It stays explicit about telemetry domains:

- raw telemetry explains what was actually observed
- time-aligned telemetry explains when something happened
- distance alignment is not inferred here when the source data does not support it

The notebook therefore treats its shape checks as a raw/time-domain
quality aid, not as authoritative gained/lost truth.


## Runtime Setup

`skrub 0.9.0` needs `SKB_DATA_DIRECTORY`, Matplotlib must use a
noninteractive backend here, and all notebook artifacts are written
under `artifacts/2025-telemetry-bad-lap-eda/`.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "notebooks").exists() and REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks import telemetry_bad_lap_support as eda

print(f"artifact dir: {eda.ARTIFACT_DIR}")
print(f"skrub data dir: {eda.SKRUB_DATA_DIR}")


artifact dir: /Users/fabio/Workspace/race-telemetry-workbench/artifacts/2025-telemetry-bad-lap-eda
skrub data dir: /Users/fabio/Workspace/race-telemetry-workbench/artifacts/2025-telemetry-bad-lap-eda/skrub-data


## Run The Analysis Once

`run_analysis()` performs the bounded 2025 race queries, classifies
lap-quality signals, writes figures and tables, and returns the data
used by the story below.


In [2]:
result = eda.run_analysis(write_outputs=True)

classified = result["classified"]
thresholds_df = result["thresholds_df"]
category_summary = result["category_summary"]
primary_summary = result["primary_summary"]
lens_summary = result["lens_summary"]
safety_summary = result["safety_summary"]
recommendation_summary = result["recommendation_summary"]
intersections = result["intersections"]
waterfall = result["waterfall"]
race_summary = result["race_summary"]
primary_by_race = result["primary_by_race"]
race_drilldowns = result["race_drilldowns"]
primary_audit = result["primary_audit"]
driver_race_matrix = result["driver_race_matrix"]
threshold_summary = result["threshold_summary"]
threshold_by_race = result["threshold_by_race"]
threshold_by_driver = result["threshold_by_driver"]
borderline_laps = result["borderline_laps"]
speed_profile_baselines = result["speed_profile_baselines"]
shape_profile_examples = result["shape_profile_examples"]
shape_cluster_exemplars = result["shape_cluster_exemplars"]
shape_cluster_stability = result["shape_cluster_stability"]
driver_summary = result["driver_summary"]
examples = result["examples"]
cluster_profile = result["cluster_profile"]
apexline_summary = result["apexline_summary"]
apexline_examples = result["apexline_examples"]

print(f"skrub version used: {result['skrub_version']}")
print(f"laps inspected: {len(classified):,}")
print(f"summary path: {result['summary_path']}")


skrub version used: 0.9.0
laps inspected: 26,689
summary path: /Users/fabio/Workspace/race-telemetry-workbench/docs/data-quality/2025-telemetry-bad-lap-eda-summary.md


In [3]:
import pandas as pd
from IPython.display import HTML, Markdown, display

def show_table(df, columns=None, sort_by=None, ascending=False, limit=10, title=None):
    table = df.copy()
    if sort_by and sort_by in table.columns:
        table = table.sort_values(sort_by, ascending=ascending)
    if columns:
        keep = [column for column in columns if column in table.columns]
        if keep:
            table = table[keep]
    if title:
        display(Markdown(f"### {title}"))
    display(table.head(limit))

def metric_row(cards):
    html_cards = []
    for label, value, detail in cards:
        html_cards.append(
            f'''
            <div style="flex:1; min-width:180px; border:1px solid #d6dde6; border-radius:14px; padding:14px 16px; background:linear-gradient(180deg,#fbfdff 0%,#f2f6fa 100%);">
              <div style="font-size:12px; text-transform:uppercase; letter-spacing:0.08em; color:#5b677a; margin-bottom:6px;">{label}</div>
              <div style="font-size:28px; font-weight:700; color:#1f2933; line-height:1.1;">{value}</div>
              <div style="font-size:13px; color:#52606d; margin-top:6px;">{detail}</div>
            </div>
            '''
        )
    display(
        HTML(
            '<div style="display:flex; gap:12px; flex-wrap:wrap; margin:10px 0 18px 0;">'
            + "".join(html_cards)
            + "</div>"
        )
    )

def callout(title: str, body: str, accent: str = "#1f6feb"):
    display(
        HTML(
            f'''
            <div style="border-left:5px solid {accent}; background:#f7fafc; padding:12px 16px; margin:10px 0 16px 0; border-radius:10px;">
              <div style="font-weight:700; color:#102a43; margin-bottom:4px;">{title}</div>
              <div style="color:#334e68; line-height:1.45;">{body}</div>
            </div>
            '''
        )
    )


## Quick Read

Start with the seasonal shape before diving into lap-level evidence.
The point here is to answer three questions quickly:

1. How much telemetry is being flagged at all?
2. Are those flags driven by a few races or spread across the season?
3. Are we mostly seeing integrity problems, context-heavy laps, or exploratory shape outliers?


In [4]:
total_laps = len(classified)
bad_laps = int(classified["bad_lap_any_category"].sum())
bad_pct = bad_laps / total_laps * 100 if total_laps else 0
top_race = race_summary.sort_values("bad_pct", ascending=False).iloc[0]
top_category = category_summary.sort_values("laps", ascending=False).iloc[0]

metric_row([
    ("Laps inspected", f"{total_laps:,}", "2025 race sessions only"),
    ("Flagged laps", f"{bad_laps:,}", f"{bad_pct:.1f}% with at least one category flag"),
    ("Most affected race", top_race["event_name"], f"{top_race['bad_pct']:.1f}% flagged"),
    ("Largest category", top_category["category"], f"{int(top_category['laps']):,} laps"),
])

callout(
    "Interpretation rule",
    "These counts are deliberately non-mutually-exclusive. A pit lap can also be sparse, context-heavy, or shape-atypical, so the figures below are about evidence layers rather than a single defect code.",
)


### Which Signals Show Up Most Often

The first chart answers what types of evidence are driving the season-wide flagged-lap count.

![Which Signals Show Up Most Often](../artifacts/2025-telemetry-bad-lap-eda/figures/category_counts.svg)


### How Uneven The Season Really Is

The second chart asks whether the issue is systemic or concentrated in a handful of race weekends.

![How Uneven The Season Really Is](../artifacts/2025-telemetry-bad-lap-eda/figures/race_bad_lap_rates.svg)


In [5]:
show_table(
    race_summary,
    columns=["event_round", "event_name", "laps", "bad_laps", "bad_pct"],
    sort_by="bad_pct",
    ascending=False,
    limit=8,
    title="Races With The Highest Flagged-Lap Rate",
)


### Races With The Highest Flagged-Lap Rate

,event_round,event_name,bad_laps,bad_pct
11,12,British Grand Prix,601,72.848485
12,13,Belgian Grand Prix,396,45.051195
0,1,Australian Grand Prix,410,44.228695
14,15,Dutch Grand Prix,432,31.671554
20,21,São Paulo Grand Prix,359,28.697042
22,23,Qatar Grand Prix,272,25.492034
21,22,Las Vegas Grand Prix,219,24.717833
16,17,Azerbaijan Grand Prix,233,24.070248


## What Is Actually Driving The Flags

After the headline counts, the next question is whether the flagged
set is dominated by cleanly separable categories or by multi-reason
laps that need a more careful review path.


### Common Reason Overlaps

This matters because a product rule built on a single flag can hide the real cause when certain combinations repeat.

![Common Reason Overlaps](../artifacts/2025-telemetry-bad-lap-eda/figures/category_intersections.svg)


### Deterministic Primary-Category Assignment

The waterfall is the reporting layer: it assigns one primary category after preserving the richer multi-flag evidence.

![Deterministic Primary-Category Assignment](../artifacts/2025-telemetry-bad-lap-eda/figures/decision_waterfall.svg)


### What The Current Rules Want The Product To Do

Recommendations stay separate from raw flags so notebook outputs do not hard-code product behavior into the schema.

![What The Current Rules Want The Product To Do](../artifacts/2025-telemetry-bad-lap-eda/figures/recommendation_summary.svg)


In [6]:
show_table(
    primary_summary,
    columns=["primary_category", "laps", "pct_laps"],
    sort_by="laps",
    ascending=False,
    limit=10,
    title="Primary Categories",
)
show_table(
    recommendation_summary,
    columns=["product_recommendation", "laps", "pct_laps"],
    sort_by="laps",
    ascending=False,
    limit=10,
    title="Recommendation Split",
)


### Primary Categories

,primary_category,laps
1,clean,21192
4,pit_lane_or_safety_car_influenced,3087
0,atypical_speed_profile,1559
3,import_or_source_data_anomaly,374
6,timing_session_boundary_artifact,369
5,position_trace_discontinuity,107
2,implausible_channel_values,1


### Recommendation Split

,product_recommendation,laps
1,keep,21192
2,keep_with_context_label,4966
0,exclude,477
3,manual_review,54


## Where The Season Concentrates

This is the part that should feel most like a human EDA: not just
how many rows are bad, but where the story visibly clusters by race,
lap progression, and driver/race combinations.


### Race-By-Race Decomposition

Some races are noisy for one dominant reason, while others are broad multi-surface problems.

![Race-By-Race Decomposition](../artifacts/2025-telemetry-bad-lap-eda/figures/race_primary_category_decomposition.svg)


### Where In A Race The Flags Show Up

A lap-number heatmap is often more revealing than another ranked table because it surfaces whether issues arrive at starts, pit cycles, or closing laps.

![Where In A Race The Flags Show Up](../artifacts/2025-telemetry-bad-lap-eda/figures/lap_number_primary_category_heatmap.svg)


### Driver/Race Concentration

This is the fastest way to see whether we have isolated driver streams or race-wide telemetry conditions.

![Driver/Race Concentration](../artifacts/2025-telemetry-bad-lap-eda/figures/driver_race_quality_matrix.svg)


In [7]:
show_table(
    driver_summary,
    columns=["driver_code", "laps", "bad_laps", "bad_pct"],
    sort_by="bad_pct",
    ascending=False,
    limit=10,
    title="Most Affected Drivers",
)
show_table(
    race_drilldowns[race_drilldowns["flagged_laps"] > 0],
    columns=["event_name", "lap_number", "primary_category", "flagged_laps"],
    sort_by="flagged_laps",
    ascending=False,
    limit=12,
    title="Selected Race Drilldowns",
)


### Most Affected Drivers

,driver_code,bad_laps,bad_pct
0,ALB,288,22.035195
16,RUS,317,21.983356
7,GAS,289,21.977186
1,ALO,275,21.842732
10,HUL,280,21.756022
18,STR,285,21.640091
17,SAI,265,21.233974
19,TSU,294,21.212121
3,BEA,291,21.194465
20,VER,290,21.090909


### Selected Race Drilldowns

,event_name,lap_number,primary_category,flagged_laps
178,Belgian Grand Prix,10,pit_lane_or_safety_car_influenced,20
177,Belgian Grand Prix,9,pit_lane_or_safety_car_influenced,20
208,Belgian Grand Prix,33,pit_lane_or_safety_car_influenced,20
207,Belgian Grand Prix,32,pit_lane_or_safety_car_influenced,20
206,Belgian Grand Prix,31,pit_lane_or_safety_car_influenced,20
205,Belgian Grand Prix,30,pit_lane_or_safety_car_influenced,20
204,Belgian Grand Prix,29,pit_lane_or_safety_car_influenced,20
203,Belgian Grand Prix,28,pit_lane_or_safety_car_influenced,20
181,Belgian Grand Prix,13,pit_lane_or_safety_car_influenced,20
180,Belgian Grand Prix,12,pit_lane_or_safety_car_influenced,20


## How Stable Are The Labels

A useful EDA should admit where the labels are fragile. This section
asks which races and laps move when the thresholds are nudged rather
than pretending the chosen cutoffs are magically exact.


### Scenario Sensitivity

Positive deltas add flagged laps versus the baseline, negative deltas remove them.

![Scenario Sensitivity](../artifacts/2025-telemetry-bad-lap-eda/figures/threshold_sensitivity.svg)


### Where Borderline Decisions Accumulate

If a race keeps showing up here, it deserves manual review before downstream tooling treats the labels as settled.

![Where Borderline Decisions Accumulate](../artifacts/2025-telemetry-bad-lap-eda/figures/borderline_laps_by_race.svg)


In [8]:
show_table(
    threshold_summary,
    columns=[
        "scenario",
        "bad_laps",
        "bad_laps_delta_vs_baseline",
        "exclude_laps",
        "context_label_laps",
    ],
    sort_by="bad_laps_delta_vs_baseline",
    ascending=False,
    limit=12,
    title="Threshold Scenarios",
)
show_table(
    borderline_laps,
    columns=[
        "event_name",
        "driver_code",
        "lap_number",
        "baseline_primary_category",
        "baseline_product_recommendation",
    ],
    sort_by="event_name",
    ascending=True,
    limit=12,
    title="Borderline Laps To Review Manually",
)


### Threshold Scenarios

,scenario,bad_laps,bad_laps_delta_vs_baseline,exclude_laps,context_label_laps
14,shape_rms_strict,5522,25,477,4966
10,max_gap_strict,5513,16,493,4966
8,p95_gap_strict,5504,7,493,4957
12,path_tolerance_strict,5502,5,516,4932
0,baseline,5497,0,477,4966
1,min_car_samples_loose,5497,0,477,4966
2,min_car_samples_strict,5497,0,477,4966
3,min_position_samples_loose,5497,0,477,4966
4,min_position_samples_strict,5497,0,477,4966
5,lap_coverage_loose,5497,0,477,4966


### Borderline Laps To Review Manually

,event_name,driver_code,lap_number,baseline_primary_category,baseline_product_recommendation
136,Abu Dhabi Grand Prix,ANT,21,clean,keep
460,Abu Dhabi Grand Prix,HAD,57,clean,keep
566,Abu Dhabi Grand Prix,HUL,48,clean,keep
121,Abu Dhabi Grand Prix,ANT,6,clean,keep
119,Abu Dhabi Grand Prix,ANT,4,clean,keep
120,Abu Dhabi Grand Prix,ANT,5,clean,keep
144,Abu Dhabi Grand Prix,ANT,29,clean,keep
1845,Australian Grand Prix,PIA,49,atypical_speed_profile,keep_with_context_label
1848,Australian Grand Prix,PIA,52,clean,keep
2069,Australian Grand Prix,VER,44,clean,keep


## What The Shape Outliers Look Like

This is the most important caveat in the notebook: shape analysis is
still equal-time and exploratory here. It can point to suspicious lap
families, but it is not a substitute for the future distance-domain
projection.


In [9]:
callout(
    "Domain caution",
    "Equal-time profile comparisons can show that two laps look different, but they do not yet tell us where time was gained or lost. That requires the distance-domain rollout.",
    accent="#b54708",
)


### Cluster Map Of Speed-Shape Families

This is the exploratory lens: it groups laps with similar equal-time profiles and quality traits.

![Cluster Map Of Speed-Shape Families](../artifacts/2025-telemetry-bad-lap-eda/figures/shape_clusters.svg)


### Representative Shape Examples

These traces are the visual answer to 'what does an outlier actually look like?'

![Representative Shape Examples](../artifacts/2025-telemetry-bad-lap-eda/figures/representative_speed_shapes.svg)


### Examples Against Same-Race Bands

The median plus 10th-90th bands make the outliers easier to trust than a raw RMS number alone.

![Examples Against Same-Race Bands](../artifacts/2025-telemetry-bad-lap-eda/figures/shape_profile_quantile_bands.svg)


### Cluster Stability Check

The clusters are only useful if they are reasonably stable under reruns and sampling variation.

![Cluster Stability Check](../artifacts/2025-telemetry-bad-lap-eda/figures/shape_cluster_stability.svg)


In [10]:
show_table(
    shape_profile_examples,
    columns=[
        "event_name",
        "driver_code",
        "lap_number",
        "comparison_group",
        "speed_profile_rms",
    ],
    sort_by="speed_profile_rms",
    ascending=False,
    limit=10,
    title="Most Extreme Shape Examples",
)
show_table(
    shape_cluster_stability,
    columns=list(shape_cluster_stability.columns),
    limit=10,
    title="Stability Metrics",
)


### Most Extreme Shape Examples

,event_name,driver_code,lap_number
16829,Italian Grand Prix,ALB,2
16830,Italian Grand Prix,ALB,3
16831,Italian Grand Prix,ALB,4
16832,Italian Grand Prix,ALB,5
16833,Italian Grand Prix,ALB,6
16834,Italian Grand Prix,ALB,7
16835,Italian Grand Prix,ALB,8
16836,Italian Grand Prix,ALB,9
16837,Italian Grand Prix,ALB,10
16838,Italian Grand Prix,ALB,11


### Stability Metrics

,k,seed,sampled_laps,silhouette,ari_vs_seed_0
0,5,0,5000,0.719351,1.000000
1,5,1,5000,0.712697,1.000000
2,5,2,5000,0.722506,1.000000
3,6,0,5000,0.727724,1.000000
4,6,1,5000,0.717030,0.999905
5,6,2,5000,0.728601,0.999895
6,7,0,5000,0.715755,1.000000
7,7,1,5000,0.716757,0.982815
8,7,2,5000,0.686226,0.943106
9,8,0,5000,0.683948,1.000000


## Evidence Tables, Not Evidence Dumps

The notebook still needs concrete rows for auditability, but the
point is to keep them short and positioned after the chart that made
them interesting.


In [11]:
show_table(
    examples,
    columns=[
        "event_name",
        "driver_code",
        "lap_number",
        "primary_category",
        "product_recommendation",
    ],
    sort_by="event_name",
    ascending=True,
    limit=15,
    title="Representative Manual-Review Examples",
)
show_table(
    primary_audit,
    columns=[
        "event_name",
        "driver_code",
        "lap_number",
        "primary_category",
        "reason_set",
    ],
    sort_by="event_name",
    ascending=True,
    limit=12,
    title="Primary Category Audit Samples",
)


### Representative Manual-Review Examples

,event_name,driver_code,lap_number,primary_category
0,Australian Grand Prix,ALB,2,timing_session_boundary_artifact
36,Australian Grand Prix,HAM,45,atypical_speed_profile
35,Australian Grand Prix,HAM,44,atypical_speed_profile
34,Australian Grand Prix,GAS,45,atypical_speed_profile
33,Australian Grand Prix,ALB,5,pit_lane_or_safety_car_influenced
32,Australian Grand Prix,ALB,4,timing_session_boundary_artifact
31,Australian Grand Prix,ALB,3,timing_session_boundary_artifact
30,Australian Grand Prix,ALB,2,timing_session_boundary_artifact
29,Australian Grand Prix,ALB,1,atypical_speed_profile
23,Australian Grand Prix,ALB,31,pit_lane_or_safety_car_influenced


### Primary Category Audit Samples

,event_name,driver_code,lap_number,primary_category
308,Australian Grand Prix,HAD,1,timing_session_boundary_artifact
250,Australian Grand Prix,DOO,1,timing_session_boundary_artifact
89,Australian Grand Prix,ALO,33,timing_session_boundary_artifact
469,Australian Grand Prix,LAW,47,timing_session_boundary_artifact
12391,Austrian Grand Prix,VER,1,timing_session_boundary_artifact
11350,Austrian Grand Prix,ANT,1,timing_session_boundary_artifact
18514,Azerbaijan Grand Prix,PIA,1,timing_session_boundary_artifact
3959,Bahrain Grand Prix,RUS,54,position_trace_discontinuity
3952,Bahrain Grand Prix,RUS,47,position_trace_discontinuity
3958,Bahrain Grand Prix,RUS,53,position_trace_discontinuity


## Geometry Cross-Check

Apexline remains the stronger geometry-reference audit. This notebook
uses it as a cross-check, not as a hidden replacement for the local
database-surface analysis.


In [12]:
if apexline_summary.empty:
    callout(
        "Apexline cross-check unavailable",
        "No Apexline summary was loaded in this environment, so geometry-reference comparison is not shown here.",
        accent="#8d2b0b",
    )
else:
    show_table(
        apexline_summary,
        columns=[
            "event_name",
            "apexline_total_laps",
            "apexline_bad_laps",
            "apexline_shape_bad_laps",
        ],
        sort_by="apexline_bad_laps",
        ascending=False,
        limit=10,
        title="Apexline Event Summary",
    )
    if not apexline_examples.empty:
        show_table(
            apexline_examples,
            columns=["event_name", "driver_code", "lap_number"],
            limit=10,
            title="Apexline Example Laps",
        )


### Apexline Event Summary

,event_name,apexline_total_laps,apexline_bad_laps,apexline_shape_bad_laps
0,Australian Grand Prix,927,359,0
11,British Grand Prix,825,330,1
14,Dutch Grand Prix,1364,323,0
6,Emilia Romagna Grand Prix,1207,251,0
8,Spanish Grand Prix,1203,213,0
12,Belgian Grand Prix,879,207,75
20,São Paulo Grand Prix,1251,202,0
9,Canadian Grand Prix,1349,192,0
3,Bahrain Grand Prix,1128,176,0
7,Monaco Grand Prix,1425,154,0


### Apexline Example Laps

,event_name,driver_code,lap_number
0,Saudi Arabian Grand Prix,HUL,25
1,British Grand Prix,ALO,24
2,Belgian Grand Prix,STR,6
3,Belgian Grand Prix,STR,19
4,Belgian Grand Prix,TSU,5
5,Belgian Grand Prix,STR,15
6,Belgian Grand Prix,HAD,10
7,Belgian Grand Prix,PIA,5
8,Belgian Grand Prix,ALO,44
9,Belgian Grand Prix,RUS,16


## Appendix: Thresholds And Supporting Lenses

The detailed thresholds still matter, but they belong after the
visual argument rather than before it.


In [13]:
show_table(thresholds_df, limit=len(thresholds_df), title="Threshold Definitions")
show_table(lens_summary, limit=len(lens_summary), title="Quality Lenses")
show_table(safety_summary, limit=len(safety_summary), title="Safety Lenses")


### Threshold Definitions

,threshold,value,rationale
0,min_car_samples,50.00,Reject laps with too little raw car-channel su...
1,min_position_samples,100.00,Reject laps with too little position support f...
2,min_lap_coverage_ratio,0.80,Flag laps where raw car telemetry covers too l...
3,path_length_tolerance_pct,0.05,Flag position paths that are materially shorte...
4,telemetry_p95_gap_ms,500.00,Flag sustained raw car telemetry cadence gaps.
5,telemetry_max_gap_ms,2000.00,Flag single large raw car telemetry gaps.
6,speed_min_kmh,0.00,Catch impossible negative speed values.
7,speed_max_kmh,380.00,Catch physically/source-improbable speed values.
8,rpm_min,0.00,Catch impossible negative RPM values.
9,rpm_max,16000.00,Catch source-improbable RPM values.


### Quality Lenses

,lens,laps,pct_of_all_laps
0,data_integrity,3988,14.942486
3,analytical_shape,3224,12.079883
2,race_context,3199,11.986212
1,replay_blocking_integrity,477,1.787253
4,manual_review,58,0.217318


### Safety Lenses

,derived_flag,value,laps,pct_of_all_laps
0,safe_for_replay,true,26212,98.212747
1,safe_for_time_domain_analysis,true,21192,79.403500
2,distance_alignment_status,not_evaluated_requires_distance_projection,26689,100.000000
3,safe_for_lap_comparison,true,21192,79.403500
4,safe_for_geometry_reference,true,21192,79.403500
5,needs_manual_review,true,58,0.217318


## Outputs

Primary written artifacts:

- `docs/data-quality/2025-telemetry-bad-lap-eda-summary.md`
- `artifacts/2025-telemetry-bad-lap-eda/skrub_lap_quality_table_report.html`
- `artifacts/2025-telemetry-bad-lap-eda/tables/*.csv`
- `artifacts/2025-telemetry-bad-lap-eda/tables/classified_laps_2025.parquet`
- `artifacts/2025-telemetry-bad-lap-eda/metadata.json`
- `artifacts/2025-telemetry-bad-lap-eda/figures/*.svg`

The explicit limitation remains unchanged: raw FastF1 `Distance` is
not part of the imported schema, so distance reset and non-monotonic
distance checks are reported as unavailable rather than guessed.
